# 🇮🇳 India Power Sector Capacity Expansion Optimization Model

**Objective:** Determine the least-cost generation mix to meet India's 2030 electricity demand while achieving the 500 GW non-fossil fuel target.

---

## About This Project

This notebook implements a **Linear Programming (LP)** optimization model to solve India's capacity expansion planning problem. The model:

- Minimizes total system cost (capital + O&M + fuel)
- Meets hourly electricity demand for 2025-2030
- Achieves 50% renewable energy generation by 2030
- Maintains 15% reserve margin for reliability
- Includes battery and pumped hydro storage

**Technologies Modeled:** Solar PV, Wind, Coal (existing + new), Gas CCGT, Large Hydro, Nuclear, Battery Storage (4hr), Pumped Hydro (8hr)

**Data Sources:** CEA, MNRE, IRENA (2024-25)

---

**Author:** [Your Name] | IIT (ISM) Dhanbad  
**Date:** January 2026

## 1. Setup and Installation

In [ ]:
# Install PuLP (other packages are pre-installed in Colab)
!pip install pulp -q

print("✅ PuLP installed!")

In [ ]:
# Clone repository from GitHub (uncomment and modify with your username)
# !git clone https://github.com/YOUR_USERNAME/india-capacity-expansion-model.git
# %cd india-capacity-expansion-model/notebooks

In [ ]:
# Import standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pulp import *
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ All packages imported successfully!")
print(f"PuLP version: {pulp.VERSION}")

In [ ]:
# Add src to path for imports
import sys
sys.path.insert(0, '..')

# Import our custom modules
from src.data_loader import DataLoader, load_data
from src.model import CapacityExpansionModel, run_scenario
from src.visualizations import ResultsVisualizer, compare_scenarios

print("✅ Custom modules imported!")

## 2. Load and Explore Data

In [ ]:
# Load all data
data = load_data('../data')

# Print summary
print(data.summary())

In [ ]:
# View demand forecast
print("\n📊 DEMAND FORECAST (CEA Projections)")
print("="*50)
display(data.demand)

In [ ]:
# View technology costs
print("\n💰 TECHNOLOGY COSTS")
print("="*50)
display(data.tech_costs)

In [ ]:
# View existing capacity
print("\n🏭 EXISTING CAPACITY (March 2025)")
print("="*50)
display(data.existing_capacity)

In [ ]:
# Plot hourly profiles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Solar and Wind profiles
ax1 = axes[0]
ax1.plot(data.hourly_profiles.index, data.hourly_profiles['solar_cf'], 
         'gold', linewidth=2, label='Solar')
ax1.plot(data.hourly_profiles.index, data.hourly_profiles['wind_cf'], 
         'green', linewidth=2, label='Wind')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Capacity Factor')
ax1.set_title('Solar and Wind Capacity Factor Profiles')
ax1.legend()
ax1.set_xticks(range(0, 24, 3))

# Demand profile
ax2 = axes[1]
ax2.plot(data.hourly_profiles.index, data.hourly_profiles['demand_factor'], 
         'red', linewidth=2)
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Demand Factor (fraction of peak)')
ax2.set_title('Typical Daily Demand Profile')
ax2.set_xticks(range(0, 24, 3))
ax2.axvline(x=18, color='gray', linestyle='--', alpha=0.5, label='Evening Peak')
ax2.legend()

plt.tight_layout()
plt.show()

print("\n💡 Key Insight: Peak demand occurs in evening (6-7 PM) when solar is declining.")

## 3. Build and Solve the Base Case Model

In [ ]:
# Base case configuration
base_config = {
    'discount_rate': 0.10,
    'reserve_margin': 0.15,
    're_target': 0.50,
}

# Create and build model
print("🔧 Building Base Case Model...")
print("="*60)

model = CapacityExpansionModel(data, base_config)
model.build()

In [ ]:
# Solve the model
print("\n🚀 Solving Optimization...")
print("="*60)

success = model.solve(solver='CBC', time_limit=300)

if success:
    print("\n✅ OPTIMIZATION SUCCESSFUL!")
else:
    print("\n❌ Optimization failed.")

In [ ]:
# Print results summary
print(model.summary())

## 4. Visualize Results

In [ ]:
# Create output directory
import os
os.makedirs('../outputs/figures', exist_ok=True)

# Initialize visualizer
viz = ResultsVisualizer(model, output_dir='../outputs/figures')

In [ ]:
# Plot capacity mix evolution
fig = viz.plot_capacity_mix(save=True)
plt.show()

In [ ]:
# Plot generation mix
fig = viz.plot_generation_mix(save=True)
plt.show()

In [ ]:
# Plot 2030 generation pie chart
fig = viz.plot_generation_pie_2030(save=True)
plt.show()

In [ ]:
# Plot new capacity additions
fig = viz.plot_new_capacity_additions(save=True)
plt.show()

In [ ]:
# Plot RE share evolution
fig = viz.plot_re_share_evolution(save=True)
plt.show()

In [ ]:
# Plot hourly dispatch for 2030
fig = viz.plot_hourly_dispatch(year=2030, save=True)
plt.show()

## 5. Export Results

In [ ]:
# Get results as DataFrames
capacity_df = model.get_capacity_df()
generation_df = model.get_generation_df()

# Save to CSV
capacity_df.to_csv('../outputs/optimal_capacity_mix.csv', index=False)
generation_df.to_csv('../outputs/generation_dispatch.csv', index=False)

print("📁 Results saved!")

# Display capacity results
print("\n📊 OPTIMAL CAPACITY MIX (MW)")
pivot_cap = capacity_df.pivot(index='technology', columns='year', values='total_capacity_mw')
display(pivot_cap.round(0))

In [ ]:
# Display generation results
print("\n⚡ GENERATION MIX (GWh)")
pivot_gen = generation_df.pivot(index='technology', columns='year', values='generation_gwh')
display(pivot_gen.round(0))

## 6. Scenario Analysis

In [ ]:
# Run High RE scenario (60% target)
print("\n" + "="*60)
print("Running High RE Scenario (60% target)...")
print("="*60)

high_re_config = {
    'discount_rate': 0.10,
    'reserve_margin': 0.15,
    're_target': 0.60,
}

high_re_model = CapacityExpansionModel(data, high_re_config)
high_re_model.build()
high_re_model.solve()

print(high_re_model.summary())

In [ ]:
# Compare scenarios
print("\n" + "="*60)
print("SCENARIO COMPARISON")
print("="*60)

comparison_data = {
    'Scenario': ['Base Case (50% RE)', 'High RE (60%)'],
    'Total Cost (₹ Cr)': [
        model.results['total_cost'],
        high_re_model.results['total_cost']
    ],
    'Solar 2030 (GW)': [
        model.results['total_capacity']['solar'][2030] / 1000,
        high_re_model.results['total_capacity']['solar'][2030] / 1000
    ],
    'RE Share 2030 (%)': [
        sum(model.results['generation_mix'][2030][t] for t in data.get_renewable_technologies()),
        sum(high_re_model.results['generation_mix'][2030][t] for t in data.get_renewable_technologies())
    ]
}

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df.round(2))

## 7. Key Findings

In [ ]:
# Generate key findings
solar_2030 = model.results['total_capacity']['solar'][2030] / 1000
wind_2030 = model.results['total_capacity']['wind'][2030] / 1000
storage_2030 = (model.results['total_capacity']['bess_4hr'][2030] + 
                model.results['total_capacity']['psh_8hr'][2030]) / 1000
re_share = sum(model.results['generation_mix'][2030][t] for t in data.get_renewable_technologies())

print("📋 KEY FINDINGS")
print("="*60)
print(f"""
1. OPTIMAL CAPACITY MIX BY 2030:
   • Solar PV: {solar_2030:.0f} GW
   • Wind: {wind_2030:.0f} GW 
   • Energy Storage: {storage_2030:.0f} GW

2. GENERATION MIX IN 2030:
   • Renewable share: {re_share:.1f}%

3. TOTAL SYSTEM COST:
   • Rs {model.results['total_cost']/1e5:.2f} Lakh Crore

4. KEY INSIGHTS:
   • Solar PV is the most cost-effective source
   • Storage is critical for evening peak
   • New coal capacity is NOT optimal
""")

## 8. Download Results

In [ ]:
# Zip and download results (for Colab)
import shutil

shutil.make_archive('../capacity_expansion_results', 'zip', '../outputs')

try:
    from google.colab import files
    files.download('../capacity_expansion_results.zip')
    print("📥 Download started!")
except:
    print("📁 Results saved to outputs/ directory")

---

## Technical Appendix

### Mathematical Formulation

**Objective:** Minimize total system cost

**Constraints:**
1. Demand balance for each hour
2. Capacity limits
3. RE target (50% by 2030)
4. Reserve margin (15%)
5. Storage energy balance

### Data Sources
- CEA: Demand projections, existing capacity
- MNRE: RE capacity and targets
- IRENA: Technology costs

### Limitations
1. Representative days (not full 8760 hours)
2. All-India aggregate model
3. Deterministic optimization

---
**End of Notebook**